##**Ejemplo de desarrollo an?lisis en DM basado en alquileres de piso**

?? **Objetivo del Proyecto:**  
Desarrollar la fase de entendimiento y preparaci?n de los datos seg?n la metodolog?a CRISP-DM, aplicando las herramientas Numpy, Pandas, Matplotlib, Seaborn, Scikit-learn y TensorFlow a la informaci?n proporcionada por el conjunto de datos ?Madrid Province Rent Data?. El objetivo consiste en realizar un estudio pormenorizado de las variables disponibles, detectar posibles inconsistencias y preparar la informaci?n para su modelado posterior.

In [ ]:
# 1. Librear?as de manipulaci?n de Datos
import numpy as np
import pandas as pd

# 2. Visualizaci?n de Datos
import seaborn as sns
import matplotlib.pyplot as plt

# 3. Preprocesamiento y Transformaciones
from sklearn.preprocessing import (
    PowerTransformer, StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer

# 4. Selecci?n de Caracter?sticas y Reducci?n de Dimensionalidad
from sklearn.feature_selection import (
    SelectKBest, f_regression, chi2, mutual_info_classif, f_classif
)
from sklearn.decomposition import PCA

# 5. Modelado y Algoritmos
from sklearn.cluster import KMeans
from sklearn.ensemble import (
    GradientBoostingRegressor, IsolationForest, RandomForestRegressor, RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC

# Separar variables predictoras y objetivo
from sklearn.calibration import label_binarize

# 6. Evaluaci?n y M?tricas
from sklearn.metrics import (
    mean_absolute_percentage_error, r2_score, mean_absolute_error, mean_squared_error,
    classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# 7. Optimizaci?n y Validaci?n
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV,
    cross_val_predict, StratifiedKFold
)

# 8. Manejo de Desequilibrio en Datos
from imblearn.over_sampling import SMOTE

# 9. Otras Utilidades
from scipy.stats import randint, uniform
import time
import joblib
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Carga inicial del Dataset

En esta secci?n, se carga el dataset 'idealista.csv' y se realiza una limpieza inicial eliminando columnas irrelevantes.
* web_id: idealista id
* url: idealista link
* title: english title
* type: property type
* price: price to rent the property
* deposit: deposit needed to rent
* private_owner: if the property owner is private or not
* professional_name: agency name, null => private owner
* floor_built: built area in square meters. Superficie total
* floor_area: living area in square meters. Superficie util
* floor: floor number
* year_built: year of the building
* orientation: orientation of the property
* bedrooms: number of bedrooms
* bathrooms: number of bathrooms
* second_hand: if the property is not new or not
* lift: if the property has lift or not
* garage_included: if the property has garage or not
* furnished: if the property is furnished or not
* equipped_kitchen: if the property has equipped kitchen or not
* fitted_wardrobes: if the property has fitted wardrobes or not
* air_conditioning: if the property has air conditioning or not
* terrace: if the property has terrace or not
* balcony: if the property has balcony or not
* storeroom: if the property has storeroom or not
* swimming_pool: if the property has swimming pool or not
* garden_area: if the property has a garden area or not
* location: property address
* district: property district from idealista
* subdistrict: property subdistrict from idealista
* postalcode: property postal code
* last update: last update date from idealista

In [ ]:
#carga de datos de "Madrid Province Rent Data" en un dataframe df
df=pd.read_csv("idealista.csv")

In [ ]:
print(df.info())

In [ ]:
# Eliminar columnas no necesarias
df = df.drop(['web_id', 'url', 'title', 'floor_area'], axis=1)
#df.dropna(subset=['price'])
df.head(2)

In [ ]:
print(df.info())

In [ ]:
#se identifica los valores nulos
df.isna().sum()

In [ ]:
#Se rellena los valores nulos
#Se considera que si floor es null entonces es porque es una casa o un chalet y no tiene pisos
df['floor'] = df['floor'].fillna(0)
df['deposit'] = df['deposit'].fillna(0)
df['professional_name'] = df['professional_name'].fillna('private owner')

In [ ]:
#valores nulos
df.isna().sum()

# An?lisis Exploratorio de Datos (EDA)

## Introducci?n
El **An?lisis Exploratorio de Datos (EDA)** es una fase fundamental en la ciencia de datos que permite comprender la distribuci?n de las variables, detectar patrones y posibles problemas en los datos. En esta secci?n, se realizan visualizaciones univariantes, bivariantes y multivariantes para analizar el dataset de idealista.csv

## Objetivos
* Identificar la distribuci?n de las variables clave.  
* Detectar valores at?picos (**outliers**).  
* Comprender relaciones entre variables para futuras etapas del modelado.  

## Tipos de An?lisis Realizados
1. **An?lisis Univariante**:  
   - Histogramas para ver la distribuci?n de variables num?ricas.  
   - Boxplots para detectar valores at?picos en variables clave como 'price'.  
   - Countplots para visualizar la frecuencia de categor?as en variables como `room_type`.  

2. **An?lisis Bivariante**:  
   - Scatterplots para analizar la relaci?n entre 'latitude', 'longitude' y 'price'.  
   - Boxplots comparativos para observar la distribuci?n de precios seg?n 'room_type' y 'neighbourhood_group'.  

3. **An?lisis Multivariante**:  
   - Matriz de correlaciones (heatmap) para ver relaciones entre variables num?ricas.  
   - Pairplots para visualizar interacciones clave entre m?ltiples variables.  

A continuaci?n, se presentan los gr?ficos con sus respectivas interpretaciones.


- An?lisis de la Distribuci?n de Precios

In [ ]:
plt.figure(figsize=(10, 5))

# Histograma de precios con KDE
plt.subplot(1, 2, 1)
sns.histplot(df["price"], bins=50, kde=True, color='b')
plt.xlim(0, df["price"].quantile(0.99))  # Limitar a percentil 99
plt.title("Distribuci?n de Precios")
plt.xlabel("Price")
plt.ylabel("Frecuencia")

# Boxplot de precios
plt.subplot(1, 2, 2)
sns.boxplot(x=df["price"], showfliers=True, color='c')
plt.xlim(0, df["price"].quantile(0.99))
plt.title("Boxplot de Precios")
plt.xlabel("Price")

plt.tight_layout()
plt.show()

## Observaciones del gr?fico

El gr?fico presentado muestra dos visualizaciones clave sobre la variable price:

1. **Histograma con KDE (Izquierda)**  

La mayor?a de los precios de los alojamientos est?n concentrados en valores bajos (entre $1000 y $2000).
La distribuci?n est? altamente sesgada a la derecha, con una larga cola de valores altos.
Existen precios extremadamente elevados que afectan la media y generan sesgo en los datos.
2. **Boxplot de price (Derecha)**

Se observa una mediana cercana a los  1000? 1500.
Hay una gran cantidad de valores at?picos (outliers), especialmente a partir de los $4000.
La presencia de estos valores extremos puede distorsionar el an?lisis y afectar modelos de predicci?n.

?? Problema Detectado
- **Sesgo en la distribuci?n**: La variable price no sigue una distribuci?n normal, lo que puede afectar modelos que asumen normalidad en los datos.

- **Valores at?picos**: Los outliers pueden impactar negativamente en el desempe?o de los modelos de regresi?n, ya que pueden generar predicciones inestables.

- **Posible soluci?n**: Se recomienda aplicar una transformaci?n logar?tmica (log(price)) para reducir la asimetr?a y mejorar la distribuci?n de los datos.

## Matriz de Correlaciones entre Variables Num?ricas

In [ ]:
# Seleccionar solo columnas num?ricas
numeric_df = df.select_dtypes(include=['number'])
# Generar el heatmap de correlaciones
plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlaciones (Solo Variables Num?ricas)")
plt.show()

##Descripci?n del gr?fico
Este heatmap muestra la matriz de correlaciones entre las variables num?ricas del dataset de idealista. Cada celda representa el coeficiente de correlaci?n entre dos variables, con un rango de valores entre -1 y 1:

- **Valores cercanos a 1**: Indican una fuerte correlaci?n positiva (cuando una variable aumenta, la otra tambi?n).
- **Valores cercanos a -1**: Indican una fuerte correlaci?n negativa (cuando una variable aumenta, la otra disminuye).
- **Valores cercanos a 0**: Indican poca o ninguna relaci?n lineal entre las variables.

In [ ]:
#Presentaci?n de resultados

# 1. Preprocesamiento de Datos Num?ricos - REGRESI?N

## **## - Evaluaci?n Inicial del Modelo: Preprocesamiento de Datos Num?ricos**
Pipeline de regresi?n

In [ ]:
#regresi?n
df.head(1)

In [ ]:
# Seleccionar columnas relevantes (sin 'price' en X)
X = df[['type', 'deposit', 'floor_built', 'private_owner', 'year_built', 'terrace',	'balcony', 'storeroom', 'swimming_pool', 'garden_area']]
# Se mantiene 'price' solo para 'y'
y = df['price']

In [ ]:
# Filtrar valores extremos en el precio (eliminar outliers del 1% superior)
df = df[df['price'] < df['price'].quantile(0.99)]

In [ ]:
#Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
numerical_columns = X_train.select_dtypes(include=[np.number]).columns # np.number alternativa
print('numerical_columns', numerical_columns)

categorical_columns = X_train.select_dtypes(exclude=[np.number]).columns
print('\ncategorical_columns',categorical_columns)

In [ ]:
# pipeline num?ricas
numerical_cols = X_train.select_dtypes(include=[np.number]).columns
pipeline_numerical = make_pipeline(
    # IterativeImputer(RandomForestRegressor(), random_state=42, initial_strategy='median'),
    SimpleImputer(strategy='median'),
)

# pipeline categ?ricas
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns
pipeline_categorical = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
)

In [ ]:
# unir pipelines con ColumnTransformer unir cols num y cat
pipeline_all = ColumnTransformer([
    ('numeric', pipeline_numerical, numerical_cols),
    ('categorical', pipeline_categorical, categorical_cols)
])

# pipeline final con el modelo
pipeline = make_pipeline(
    pipeline_all,
    # PCA(),
    RandomForestRegressor()
)
pipeline

In [ ]:
#Entrenar el modelo en train
pipeline.fit(X_train, y_train)

# Predicciones en test
y_pred = pipeline.predict(X_test)
print(f": {y_pred}")

In [ ]:
#C?lculo de m?tricas
r2_test = r2_score(y_test, y_pred)

# Mostrar resultados
print(f" R? en Test: {r2_test:.5f}")

##2. Preprocesamiento de Datos Num?ricos - CLASIFICACI?N
Pipeline de clasificaci?n

In [ ]:
# Seleccionar columnas relevantes
caracteristicas = ["price", "floor_built", "bathrooms", "year_built"]
clases = "district"


In [ ]:
# Eliminar filas con valores nulos en estas columnas
df_clean = df.dropna(subset=caracteristicas + [clases])

In [ ]:
# Identificaci?n de columnas num?ricas y categ?ricas
num_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

In [ ]:
# Preprocesador para imputaci?n, escalado y codificaci?n
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ]), cat_features)
])

# Aplicar preprocesamiento una sola vez y guardar los datos transformados
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
# Codificar la variable categ?rica "district" a valores num?ricos
#label_encoder = LabelEncoder()
#df_clean[clases] = label_encoder.fit_transform(df_clean[clases])

In [ ]:
# Separar variables predictoras (X) y variable objetivo (y)
X = df_clean[caracteristicas]
y = df_clean[clases]

In [ ]:
# Dividir en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Modelos a evaluar
models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVC': SVC(kernel='rbf', C=1, gamma='scale', probability=True)
}

In [ ]:
# Evaluaci?n con validaci?n cruzada
results = []

# Binarizar etiquetas para AUC
y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_bin.shape[1]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Crear y entrenar el modelo de clasificaci?n multiclase
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
# Realizar predicciones
y_pred = clf.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
# Evaluar el modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
print(f"Precisi?n del modelo: {accuracy:.2f}")
print("Reporte de clasificaci?n:")
#print(report)